## Task 1: Data Loading, Cleaning & Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score,accuracy_score,roc_auc_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

### 1.1 - Load and inspect the dataset

In [ ]:
df = pd.read_csv('../Dataset/adult.csv')

print("Dataset Shape:", df.shape)

print("\nDataset Data Types:")
print(df.dtypes)

print("\nFirst 5 Rows:")
print(df.head())

for col in df.select_dtypes('object'):
    df[col] = df[col].str.strip()

### 1.2 - Handle Missing Values

In [ ]:
df.replace('?', np.nan, inplace=True)

print("Missing Values:")
print(df.isnull().sum())

In [ ]:
print("Before dropping:", df.shape)

df.dropna(inplace=True)

print("After dropping:", df.shape)

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())

In [ ]:
print("Final Dataset Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())

### 1.3 - Drop redundant columns

In [ ]:
df.drop(columns=['fnlwgt', 'education'], inplace=True)

print("Updated Dataset Shape:", df.shape)

print("\nRemaining Columns:")
print(df.columns.tolist())

### 1.4 Encode the target

In [ ]:
df['income'] = (df['income'] == '>50K').astype(int)

print(df['income'].value_counts())

In [ ]:
print("Income Class Distribution:")
print(df['income'].value_counts())

print("\nIncome Class Percentage:")
print(df['income'].value_counts(normalize=True) * 100)

In [ ]:
plt.figure(figsize=(7, 5))

ax = sns.countplot(data=df, x='income', hue='income', palette='viridis', legend=False)

for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3, fontsize=11)

plt.title('Income Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Income (0 = <=50K, 1 = >50K)')
plt.ylabel('Count')
plt.xticks([0, 1], ['<=50K (0)', '>50K (1)'])
plt.tight_layout()
plt.show()

In [ ]:
print("Dataset Shape:", df.shape)
print("\nTarget Distribution:")
print(df['income'].value_counts())

print("\nTarget Distribution (%):")
print((df['income'].value_counts(normalize=True) * 100).round(2))

### 1.5 - Exploratory visuals

In [ ]:
plt.figure(figsize=(9, 5))

sns.histplot(data=df,x='age',hue='income',bins=30,kde=True,element='step',stat='count',common_norm=False,palette='viridis')

plt.title('Age Distribution by Income', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.legend(title='Income', labels=['>50K (1)', '<=50K (0)'])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(data=df,x='income',y='hours-per-week',hue='income',palette='viridis',legend=False)

plt.title('Working Hours per Week by Income', fontsize=14, fontweight='bold')
plt.xlabel('Income (0 = <=50K, 1 = >50K)')
plt.ylabel('Hours per Week')
plt.xticks([0, 1], ['<=50K (0)', '>50K (1)'])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

sns.countplot(data=df,x='educational-num',hue='income',palette='viridis')

plt.title('Education Level by Income', fontsize=14, fontweight='bold')
plt.xlabel('Education Number')
plt.ylabel('Count')
plt.legend(title='Income', labels=['<=50K (0)', '>50K (1)'])
plt.tight_layout()
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

corr_matrix = numeric_df.corr()

plt.figure(figsize=(10, 7))

sns.heatmap(corr_matrix,annot=True,cmap='coolwarm',fmt='.2f',linewidths=0.5,square=True,cbar=True)

plt.title('Correlation Heatmap of Numerical Features',fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### Overall, education level, age, working hours, and capital-related features may provide useful information for predicting income. However, correlation does not imply causation, and weaker correlations do not necessarily mean that a feature is unimportant for a classification model.

### 1.6 - Encode categoricals & scale numericals

In [ ]:
X = df.drop('income', axis=1)
y = df['income']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
X = pd.get_dummies(X, drop_first=True)

print("Shape after One-Hot Encoding:", X.shape)
print("\nTotal Features after OHE:", X.shape[1])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("Training Features Shape:", X_train.shape)
print("Testing Features Shape:", X_test.shape)

print("Training Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

In [ ]:
numeric_cols = ['age','educational-num','capital-gain','capital-loss','hours-per-week']

scaler = StandardScaler()

X_train.loc[:, numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

X_test.loc[:, numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Numerical features scaled successfully!")

In [ ]:
print("Original Class Distribution:")
print(y.value_counts(normalize=True) * 100)

print("\nTraining Class Distribution:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting Class Distribution:")
print((y_test.value_counts(normalize=True) * 100).round(2))

## Task 2: MLP Architecture - Baseline ANN

### 2.1 - Define the reusable build_ann() function

In [ ]:
def build_ann(
    input_dim,
    hidden_units=[128, 64],
    activation='relu',
    initializer='glorot_uniform',
    use_batch_norm=False,
    optimizer='adam',
    loss='binary_crossentropy'
):

    model = keras.Sequential()

    model.add(
        layers.Input(shape=(input_dim,))
    )

    for units in hidden_units:

        model.add(
            layers.Dense(
                units,
                activation=None if use_batch_norm else activation,
                kernel_initializer=initializer
            )
        )

        if use_batch_norm:
            model.add(layers.BatchNormalization())
            model.add(layers.Activation(activation))

    model.add(
        layers.Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.F1Score(name='f1_score', threshold=0.5)
        ]
    )

    return model

### 2.2 - Baseline ANN Architecture

In [ ]:
input_dim = X_train.shape[1]

baseline_model = build_ann(input_dim=input_dim)

baseline_model.summary()

### Why not Softmax?

Softmax is commonly used when predicting multiple mutually exclusive classes. For binary classification, it is possible to use two output neurons with softmax activation, but this requires additional parameters.

A single sigmoid output neuron is more efficient because it directly predicts the probability of the positive class, while the probability of the negative class can be calculated as (1-p).

Therefore, sigmoid with a single output neuron is preferred for this binary income classification problem because it provides a simple, computationally efficient output representation.

### 2.3 - Train the baseline

In [ ]:
model_base = build_ann(input_dim=X_train.shape[1])

history_base = model_base.fit(X_train,y_train,epochs=50,batch_size=64,validation_split=0.1,verbose=1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))


ax[0].plot(history_base.history['loss'], label='Training Loss', linewidth=2)
ax[0].plot(history_base.history['val_loss'], label='Validation Loss', linewidth=2)

ax[0].set_title('Training vs Validation Loss', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(history_base.history['accuracy'], label='Training Accuracy', linewidth=2)
ax[1].plot(history_base.history['val_accuracy'], label='Validation Accuracy', linewidth=2)

ax[1].set_title('Training vs Validation Accuracy', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy')
ax[1].legend()
ax[1].grid(alpha=0.3)


fig.suptitle(
    'Baseline ANN - Training Curves (ReLU - Glorot Uniform - Adam - BCE)',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

### 2.4 - Evaluate the baseline with full metrics

In [ ]:
y_pred_prob = model_base.predict(X_test).flatten()

y_pred = (y_pred_prob > 0.5).astype(int)

print("First 10 Predicted Probabilities:")
print(y_pred_prob[:10])

print("\nFirst 10 Predicted Classes:")
print(y_pred[:10])

In [ ]:
print(classification_report(y_test,y_pred,target_names=['<=50K (0)', '>50K (1)'],digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    linewidths=0.5,
    xticklabels=['<=50K (0)', '>50K (1)'],
    yticklabels=['<=50K (0)', '>50K (1)']
)

plt.title('Baseline ANN - Confusion Matrix',
          fontsize=14, fontweight='bold')

plt.xlabel('Predicted Income')
plt.ylabel('Actual Income')

plt.tight_layout()
plt.show()

In [ ]:
base_accuracy = accuracy_score(y_test, y_pred)

base_precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)

base_recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)

base_f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)

print("BASELINE ANN - TEST PERFORMANCE")
print("--------------------------------")
print(f"Accuracy  : {base_accuracy:.4f}")
print(f"Precision : {base_precision:.4f}")
print(f"Recall    : {base_recall:.4f}")
print(f"F1-Score  : {base_f1:.4f}")

In [ ]:
baseline_results = {
    'Model': 'Baseline ANN',
    'Accuracy': base_accuracy,
    'Precision': base_precision,
    'Recall': base_recall,
    'F1-Score': base_f1
}

print(baseline_results)

### Baseline ANN - Classification Performance Evaluation

The baseline ANN achieved an overall test accuracy of 84.20%, indicating that the model correctly classified approximately 84% of the test observations.

For the minority class (income >50K), the model achieved a Precision of 70.92%, meaning that approximately 71% of individuals predicted to earn more than 50K actually belong to this income group.

The Recall of 61.46% indicates that the model correctly identified approximately 61% of all individuals who actually earn more than 50K. This suggests that the model is still missing a considerable number of positive-class observations.

The F1-score of 65.85% represents the balance between Precision and Recall for the minority class.

Overall, the baseline ANN provides reasonable classification accuracy, but its Recall and F1-score for the minority class indicate room for improvement. The model could be improved to identify more individuals earning above 50K while maintaining a good balance between Precision and Recall.

### 2.5 Why tabular data suits ANN - Markdown

### Why Use a Fully Connected MLP for Structured Tabular Data?

Our Adult Income dataset contains numerical and one-hot encoded categorical features, making it suitable for an MLP architecture.

Each input neuron receives one feature from the dataset. The hidden layers learn weighted combinations of these features and identify complex non-linear relationships and interactions between them.

For example:
- Age × Education: The relationship between an individual's age and educational qualification may influence their income.
- Working Hours × Occupation: The effect of working hours on income may vary depending on the individual's occupation.

A simple Logistic Regression model primarily learns linear relationships between input features and the target variable. In contrast, an MLP with multiple hidden layers and non-linear activation functions such as ReLU can learn more complex patterns and feature interactions.

Therefore, a fully connected MLP is an appropriate choice for predicting income from structured demographic and employment-related features.

## Task 3 : Activation Functions - Comparison Experiment

### 3.1 - Tain 4 models with different hidden activations

In [ ]:
# 1. ReLU Activation
model_relu = build_ann(
    input_dim=X_train.shape[1],
    activation='relu'
)

history_relu = model_relu.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# 2. Tanh Activation

model_tanh = build_ann(
    input_dim=X_train.shape[1],
    activation='tanh'
)

history_tanh = model_tanh.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# 3. Sigmoid Activation

model_sigmoid = build_ann(
    input_dim=X_train.shape[1],
    activation='sigmoid'
)

history_sigmoid = model_sigmoid.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# 4. ELU Activation

model_elu = build_ann(
    input_dim=X_train.shape[1],
    activation='elu'
)

history_elu = model_elu.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

### 3.2 - Activations comparison - 4 panel plot

In [ ]:
activation_runs = [
    ('ReLU', history_relu),
    ('Tanh', history_tanh),
    ('Sigmoid', history_sigmoid),
    ('ELU', history_elu)
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, history) in zip(axes.flat, activation_runs):
    epochs = np.arange(1, len(history.history['accuracy']) + 1)
    final_val_accuracy = float(history.history['val_accuracy'][-1])
    final_val_f1 = float(np.asarray(history.history['val_f1_score'][-1]).reshape(-1)[0])

    ax.plot(epochs, history.history['accuracy'], color='#CC0000', label='Training Accuracy', linewidth=2)
    ax.plot(epochs, history.history['val_accuracy'], color='#2C2C2C', label='Validation Accuracy', linewidth=2)
    ax.axhline(0.75, color='gray', linestyle='--', linewidth=1.4, label='75% Majority Baseline')
    ax.set_title(f'{name} | Final Val Acc: {final_val_accuracy:.4f} | Class-1 F1: {final_val_f1:.4f}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.65, 1.0)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle('Activation Function Comparison — Training vs Validation Accuracy', fontsize=14)
plt.tight_layout()
plt.show()

### 3.3 - Dead neuron check - RELU

In [ ]:
first_hidden_dense = next(
    layer for layer in model_relu.layers
    if isinstance(layer, keras.layers.Dense)
)

intermediate = keras.Model(
    inputs=model_relu.inputs,
    outputs=first_hidden_dense.output
)

activations = intermediate.predict(
    X_test.iloc[:500],
    verbose=0
)

dead_fraction = np.mean(activations == 0)

dead_neuron_fraction = np.mean(
    np.all(activations == 0, axis=0)
)

mean_activations = np.mean(activations, axis=0)

print(f'Zero activation fraction: {dead_fraction:.4f}')
print(f'Zero activation percentage: {dead_fraction * 100:.2f}%')

print(f'Completely inactive neurons: {dead_neuron_fraction * 100:.2f}%')

plt.figure(figsize=(10, 6))

plt.hist(
    mean_activations,
    bins=20,
    color='#CC0000',
    edgecolor='black',
    alpha=0.8
)

plt.title('ReLU First-Layer Activation Distribution', fontsize=14)
plt.xlabel('Mean Activation per Neuron')
plt.ylabel('Number of Neurons')
plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

### 3.4 : Gradient flow check - Sigmoid hidden layers

In [ ]:
layer_names = ['Layer 1', 'Layer 2', 'Output']

batch_size = 64

X_batch = tf.convert_to_tensor(
    X_train.iloc[:batch_size], dtype=tf.float32
)

y_batch = tf.convert_to_tensor(
    y_train.iloc[:batch_size].values.reshape(-1, 1),
    dtype=tf.float32
)

with tf.GradientTape() as tape:
    predictions = model_sigmoid(X_batch, training=True)
    loss = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_batch, predictions)
    )

gradients = tape.gradient(
    loss, model_sigmoid.trainable_variables
)

mean_gradients = []

for i, layer in enumerate(
    layer for layer in model_sigmoid.layers
    if isinstance(layer, tf.keras.layers.Dense)
):
    kernel_gradient = gradients[2 * i]
    bias_gradient = gradients[2 * i + 1]

    mean_abs_gradient = np.mean([
        tf.reduce_mean(tf.abs(kernel_gradient)).numpy(),
        tf.reduce_mean(tf.abs(bias_gradient)).numpy()
    ])

    mean_gradients.append(mean_abs_gradient)

print(dict(zip(layer_names, mean_gradients)))

In [ ]:
plt.figure(figsize=(9, 5))

plt.barh(
    layer_names,
    mean_gradients,
    color='#CC0000',
    edgecolor='black',
    alpha=0.85
)

plt.xlabel('Mean Absolute Gradient')
plt.ylabel('Dense Layer')

plt.title(
    'Gradient Magnitude by Layer - Sigmoid Hidden Activations',
    fontsize=13
)

plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Task 4 -  Weight Initialization Techniques

### 4.1 - Train 5 models with different intializers

In [ ]:
initializers = [
    'glorot_uniform',
    'glorot_normal',
    'he_uniform',
    'he_normal',
    'zeros'
]

initializer_models = {}
initializer_histories = {}

for initializer in initializers:

    print(f"\nTraining model with initializer: {initializer}")

    model = build_ann(
        input_dim=X_train.shape[1],
        activation='relu',
        initializer=initializer
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )

    initializer_models[initializer] = model
    initializer_histories[initializer] = history

In [ ]:
for initializer, history in initializer_histories.items():
    print(
        f"{initializer}: "
        f"{len(history.history['loss'])} epochs completed"
    )

### 4.2 - Why Intialization matters - Markdown  

### Understanding Weight Initialization Techniques

Neural networks trained with all-zero weights suffer from the symmetry problem. Every neuron in a layer produces the same output, receives the same gradient, and updates to the same value. As a result, the neurons learn identical features, and the network effectively collapses to a single effective neuron per layer, regardless of the layer's width.

1. Xavier / Glorot Initialization

Xavier initialization draws weights from a distribution designed to maintain stable activation and gradient variance across layers.

It is particularly suitable for activation functions such as tanh and sigmoid because it helps prevent activations and gradients from becoming excessively large or small across layers.

2. He Initialization

He initialization uses a larger variance:

\[
Var(W)=\frac{2}{fan_{in}}
\]

The extra factor of 2 compensates approximately for ReLU setting negative activations to zero. This helps maintain the effective variance of activations and gradients in ReLU-based networks.

3. Comparison in This Experiment

- glorot_uniform: Xavier initialization using a uniform distribution.
- glorot_normal: Xavier initialization using a normal distribution.
- he_uniform: He initialization using a uniform distribution.
- he_normal: He initialization using a normal distribution.
- zeros: Initializes all weights to zero, allowing us to observe the symmetry problem.

Since our ANN uses ReLU hidden activations, He initialization is theoretically well-suited to this experiment. We will compare all five methods using the same architecture and training settings to evaluate their actual performance on the Adult Income dataset.

### 4.3 - Convergence speed comparison plot

In [ ]:
import matplotlib.pyplot as plt

styles = {
    'glorot_uniform': {'color': 'red', 'linestyle': '-'},
    'glorot_normal':  {'color': 'green', 'linestyle': '--'},
    'he_uniform':     {'color': 'blue', 'linestyle': '-.'},
    'he_normal':      {'color': 'yellow', 'linestyle': ':'},
    'zeros':          {'color': 'black', 'linestyle': '-'}
}

plt.figure(figsize=(13, 7))


for initializer in initializers:

    val_accuracy = initializer_histories[initializer].history['val_accuracy']

    epochs = range(1, len(val_accuracy) + 1)

    plt.plot(
        epochs,
        val_accuracy,
        label=initializer,
        color=styles[initializer]['color'],
        linestyle=styles[initializer]['linestyle'],
        linewidth=2
    )

    if initializer != 'zeros':

        crossed_epochs = [
            epoch for epoch, accuracy in zip(epochs, val_accuracy)
            if accuracy >= 0.82
        ]

        if crossed_epochs:
            first_epoch = crossed_epochs[0]

            plt.axvline(
                x=first_epoch,
                color=styles[initializer]['color'],
                linestyle=':',
                alpha=0.65,
                linewidth=1.5
            )

            plt.text(
                first_epoch,
                0.765,
                f'{first_epoch}',
                color=styles[initializer]['color'],
                fontsize=9,
                ha='center'
            )

plt.axhline(
    y=0.84,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='84% Target Benchmark'
)

plt.title(
    'Weight Initialisation — Convergence Speed Comparison (Adult Income Dataset)',
    fontsize=13
)

plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')

plt.xticks(range(0, 51, 5))
plt.ylim(0.70, 1.00)

plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(title='Weight Initialiser', loc='best')

plt.tight_layout()
plt.show()

### 4.4 - Zeroes failure demonstration

In [ ]:
history_zeros = initializer_histories['zeros']

epochs = range(1, len(history_zeros.history['loss']) + 1)

fig, ax1 = plt.subplots(figsize=(10, 6))


ax1.plot(
    epochs,
    history_zeros.history['loss'],
    color='#CC0000',
    linestyle='-',
    linewidth=2,
    label='Training Loss'
)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss', color='#CC0000')
ax1.tick_params(axis='y', labelcolor='#CC0000')


ax2 = ax1.twinx()

ax2.plot(
    epochs,
    history_zeros.history['val_accuracy'],
    color='#2C2C2C',
    linestyle='--',
    linewidth=2,
    label='Validation Accuracy'
)

ax2.set_ylabel('Validation Accuracy', color='#2C2C2C')
ax2.tick_params(axis='y', labelcolor='#2C2C2C')


plt.title(
    'Zero Initialization - Training Loss vs Validation Accuracy',
    fontsize=13
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc='best'
)

ax1.grid(axis='both', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

### 4.5 - Weight distribution visualisation

In [ ]:
model_he = build_ann(
    input_dim=X_train.shape[1],
    activation='relu',
    initializer='he_normal'
)

model_glorot = build_ann(
    input_dim=X_train.shape[1],
    activation='relu',
    initializer='glorot_uniform'
)

_ = model_he(tf.convert_to_tensor(X_train.iloc[:1], dtype=tf.float32))
_ = model_glorot(tf.convert_to_tensor(X_train.iloc[:1], dtype=tf.float32))


he_first_dense = next(
    layer for layer in model_he.layers
    if isinstance(layer, tf.keras.layers.Dense)
)

glorot_first_dense = next(
    layer for layer in model_glorot.layers
    if isinstance(layer, tf.keras.layers.Dense)
)


he_weights_before = he_first_dense.get_weights()[0].flatten().copy()
glorot_weights_before = glorot_first_dense.get_weights()[0].flatten().copy()


history_he = model_he.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

history_glorot = model_glorot.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)


he_weights_after = he_first_dense.get_weights()[0].flatten().copy()
glorot_weights_after = glorot_first_dense.get_weights()[0].flatten().copy()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))


axes[0, 0].hist(
    he_weights_before,
    bins=40,
    color='#CC0000',
    edgecolor='black',
    alpha=0.8
)
axes[0, 0].set_title('He Normal — Before Training')
axes[0, 0].set_xlabel('Weight Value')
axes[0, 0].set_ylabel('Frequency')


axes[0, 1].hist(
    he_weights_after,
    bins=40,
    color='#CC0000',
    edgecolor='black',
    alpha=0.8
)
axes[0, 1].set_title('He Normal — After Training')
axes[0, 1].set_xlabel('Weight Value')
axes[0, 1].set_ylabel('Frequency')


axes[1, 0].hist(
    glorot_weights_before,
    bins=40,
    color='#1F77B4',
    edgecolor='black',
    alpha=0.8
)
axes[1, 0].set_title('Glorot Uniform — Before Training')
axes[1, 0].set_xlabel('Weight Value')
axes[1, 0].set_ylabel('Frequency')


axes[1, 1].hist(
    glorot_weights_after,
    bins=40,
    color='#1F77B4',
    edgecolor='black',
    alpha=0.8
)
axes[1, 1].set_title('Glorot Uniform — After Training')
axes[1, 1].set_xlabel('Weight Value')
axes[1, 1].set_ylabel('Frequency')

fig.suptitle(
    'Weight Distributions — Initial vs Trained',
    fontsize=15,
    y=1.02
)

plt.tight_layout()
plt.show()

## Task 5 - Loss Functions

### 5.1 - Binary cross entropy - correct loss

In [ ]:
print("Baseline Loss Function:", model_base.loss)

### 5.2 - MSE as Classification loss - wrong tool experiment

In [ ]:
model_mse = build_ann(
    input_dim=X_train.shape[1],
    loss='mean_squared_error',
    activation='relu',
    initializer='he_normal'
)

history_mse = model_mse.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
epochs = range(1, 51)

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    history_base.history['val_f1_score'],
    label='Baseline ANN (Binary Crossentropy)',
    color='#CC0000',
    linewidth=2
)

plt.plot(
    epochs,
    history_mse.history['val_f1_score'],
    label='MSE Loss',
    color='#2C2C2C',
    linewidth=2
)

plt.xlabel('Epochs')
plt.ylabel('Validation F1-Score (Class 1)')
plt.title('Validation F1-Score: BCE vs MSE Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 - Weighted Binary Cross Entropy - Handling class imbalance

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1]),
    y=y_train
)

class_weight_dict = {
    0: weights[0],
    1: weights[1]
}

print("Class weights:", class_weight_dict)

In [ ]:
model_wbce = build_ann(
    input_dim=X_train.shape[1],
    loss='binary_crossentropy',
    activation='relu',
    initializer='he_normal'
)

history_wbce = model_wbce.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
epochs = range(1, 51)

metrics = {
    'Precision': 'val_precision',
    'Recall': 'val_recall',
    'F1-Score': 'val_f1_score'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (metric_name, history_key) in zip(axes, metrics.items()):

    ax.plot(
        epochs,
        history_base.history[history_key],
        label='Baseline (Unweighted BCE)',
        color='#CC0000',
        linewidth=2
    )

    ax.plot(
        epochs,
        history_wbce.history[history_key],
        label='Weighted BCE',
        color='#2C2C2C',
        linewidth=2
    )

    ax.set_title(f'Validation {metric_name} — Class 1')
    ax.set_xlabel('Epochs')
    ax.set_ylabel(metric_name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle(
    'Baseline vs Weighted BCE — Minority Class Performance',
    fontsize=14
)
plt.tight_layout()
plt.show()

### 5.4 - Focal loss - Advanced imbalance handling

In [ ]:
def focal_loss(gamma=2.0, alpha=0.25):

    def loss(y_true, y_pred):

        y_true = tf.cast(y_true, tf.float32)

        y_pred = tf.clip_by_value(
            y_pred, 1e-7, 1 - 1e-7
        )

        bce = (
            -y_true * tf.math.log(y_pred)
            - (1 - y_true) * tf.math.log(1 - y_pred)
        )

        weight = (
            alpha * y_true * (1 - y_pred) ** gamma
            + (1 - alpha) * (1 - y_true) * y_pred ** gamma
        )

        return tf.reduce_mean(weight * bce)

    return loss

In [ ]:
model_focal = build_ann(
    input_dim=X_train.shape[1],
    loss=focal_loss(gamma=2.0, alpha=0.25),
    activation='relu',
    initializer='he_normal'
)

history_focal = model_focal.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
epochs = range(1, 51)

models_history = {
    'Baseline BCE': history_base,
    'Weighted BCE': history_wbce,
    'Focal Loss': history_focal
}

metrics = {
    'Recall': 'val_recall',
    'F1-Score': 'val_f1_score'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {
    'Baseline BCE': '#CC0000',
    'Weighted BCE': '#2C2C2C',
    'Focal Loss': '#1F77B4'
}

for ax, (metric_name, history_key) in zip(axes, metrics.items()):

    for model_name, history in models_history.items():

        ax.plot(
            epochs,
            history.history[history_key],
            label=model_name,
            color=colors[model_name],
            linewidth=2
        )

    ax.set_title(f'Validation {metric_name} — Class 1 (>50K)')
    ax.set_xlabel('Epochs')
    ax.set_ylabel(metric_name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle(
    'Class-1 Performance: Baseline BCE vs Weighted BCE vs Focal Loss',
    fontsize=14
)

plt.tight_layout()
plt.show()

## Task 6 - Batch Normalization

### 6.1 - Build ANN with batch normalization

In [ ]:
model_bn = keras.Sequential([
    keras.layers.Dense(
        128,
        kernel_initializer='he_normal',
        input_shape=(X_train.shape[1],)
    ),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),

    keras.layers.Dense(
        64,
        kernel_initializer='he_normal'
    ),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),

    keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
model_bn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model_bn.summary()

### 6.3 - Train and compare : with v/s without batch normalization

In [ ]:
history_bn = model_bn.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
epochs = range(1, 51)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(
    epochs, history_base.history['loss'],
    color='#CC0000', linestyle='-',
    label='Baseline - Training Loss'
)

axes[0].plot(
    epochs, history_base.history['val_loss'],
    color='#CC0000', linestyle='--',
    label='Baseline - Validation Loss'
)

axes[0].plot(
    epochs, history_bn.history['loss'],
    color='#2C2C2C', linestyle='-',
    label='BatchNorm - Training Loss'
)

axes[0].plot(
    epochs, history_bn.history['val_loss'],
    color='#2C2C2C', linestyle='--',
    label='BatchNorm - Validation Loss'
)

axes[0].set_title('Loss Comparison')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Binary Crossentropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)


axes[1].plot(
    epochs, history_base.history['accuracy'],
    color='#CC0000', linestyle='-',
    label='Baseline - Training Accuracy'
)

axes[1].plot(
    epochs, history_base.history['val_accuracy'],
    color='#CC0000', linestyle='--',
    label='Baseline - Validation Accuracy'
)

axes[1].plot(
    epochs, history_bn.history['accuracy'],
    color='#2C2C2C', linestyle='-',
    label='BatchNorm - Training Accuracy'
)

axes[1].plot(
    epochs, history_bn.history['val_accuracy'],
    color='#2C2C2C', linestyle='--',
    label='BatchNorm - Validation Accuracy'
)

axes[1].set_title('Accuracy Comparison')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(
    'Batch Normalization vs Baseline - Training Dynamics (Adult Income)',
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
y_pred_base = (model_base.predict(X_test, verbose=0) >= 0.5).astype(int).ravel()

y_pred_bn = (model_bn.predict(X_test, verbose=0) >= 0.5).astype(int).ravel()

base_accuracy = accuracy_score(y_test, y_pred_base)
base_f1 = f1_score(y_test, y_pred_base, pos_label=1)

bn_accuracy = accuracy_score(y_test, y_pred_bn)
bn_f1 = f1_score(y_test, y_pred_bn, pos_label=1)

bn_comparison = pd.DataFrame({
    'Model': ['Baseline ANN', 'Batch Normalization ANN'],
    'Test Accuracy': [base_accuracy, bn_accuracy],
    'Test F1-Score (Class 1)': [base_f1, bn_f1]
})

print(bn_comparison.round(4))

### 6.4 - BatchNorm position experiment

In [ ]:
# (a) Batch Normalization after activation
model_bn_after = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),

    keras.layers.Dense(128, kernel_initializer='he_normal'),
    keras.layers.Activation('relu'),
    keras.layers.BatchNormalization(),

    keras.layers.Dense(64, kernel_initializer='he_normal'),
    keras.layers.Activation('relu'),
    keras.layers.BatchNormalization(),

    keras.layers.Dense(1, activation='sigmoid')
])

model_bn_after.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


# (b) Batch Normalization before activation (canonical)
model_bn_before = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),

    keras.layers.Dense(128, kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),

    keras.layers.Dense(64, kernel_initializer='he_normal'),
    keras.layers.BatchNormalization(),
    keras.layers.Activation('relu'),

    keras.layers.Dense(1, activation='sigmoid')
])

model_bn_before.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_bn_after = model_bn_after.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

history_bn_before = model_bn_before.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
epochs = range(1, 31)

plt.figure(figsize=(10, 6))

plt.plot(
    epochs,
    history_bn_after.history['val_accuracy'],
    color='#CC0000',
    linewidth=2,
    label='BN After Activation'
)

plt.plot(
    epochs,
    history_bn_before.history['val_accuracy'],
    color='#2C2C2C',
    linewidth=2,
    label='BN Before Activation (Canonical)'
)

plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.title('Batch Normalization Order — Validation Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
after_val_acc = history_bn_after.history['val_accuracy'][-1]
before_val_acc = history_bn_before.history['val_accuracy'][-1]

print(f"BN After Activation:  {after_val_acc:.4f}")
print(f"BN Before Activation: {before_val_acc:.4f}")

if after_val_acc > before_val_acc:
    print("BN After Activation performed better.")
elif before_val_acc > after_val_acc:
    print("BN Before Activation performed better.")
else:
    print("Both models achieved the same validation accuracy.")

### 6.5 - Inspect learned gamma and beta

In [ ]:
bn_layer = model_bn.layers[1]

gamma, beta, running_mean, running_var = bn_layer.get_weights()

print("Gamma shape:", gamma.shape)
print("Beta shape:", beta.shape)
print("Running Mean shape:", running_mean.shape)
print("Running Variance shape:", running_var.shape)

In [ ]:
neurons = np.arange(128)

fig, axes = plt.subplots(2, 1, figsize=(16, 9))


axes[0].bar(neurons, gamma, color='#CC0000', alpha=0.85)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Batch Normalization - Gamma Values')
axes[0].set_xlabel('Neuron Index')
axes[0].set_ylabel('Gamma')
axes[0].set_xticks(np.arange(0, 128, 4))
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(neurons, beta, color='#2C2C2C', alpha=0.85)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Batch Normalization - Beta Values')
axes[1].set_xlabel('Neuron Index')
axes[1].set_ylabel('Beta ')
axes[1].set_xticks(np.arange(0, 128, 4))
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle(
    'Learned Batch Normalization Parameters - First Hidden Layer',
    fontsize=14
)

plt.tight_layout()
plt.show()

## Task 7 : Optimizers - Convergence & Final Comparison

### 7.1 - Build 5 models with different optimisers

In [ ]:
optimizers = {
    'SGD': 'sgd',

    'SGD_Momentum': keras.optimizers.SGD(
        learning_rate=0.01,
        momentum=0.9
    ),

    'RMSprop': 'rmsprop',

    'Adam_Default': 'adam',

    'Adam_Explicit': keras.optimizers.Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999
    )
}

In [ ]:
optimizer_models = {}
optimizer_histories = {}

for optimizer_name, optimizer in optimizers.items():

    print(f"\nTraining model with {optimizer_name} optimizer...")

    model = build_ann(
        input_dim=X_train.shape[1],
        activation='relu',
        initializer='he_normal',
        use_batch_norm=True,
        loss='binary_crossentropy',
        optimizer=optimizer
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )

    optimizer_models[optimizer_name] = model
    optimizer_histories[optimizer_name] = history

### 7.2 - Optimiser convergence plot

In [ ]:
epochs = np.arange(1, 51)

styles = {
    'SGD': {
        'color': '#CC0000',
        'linestyle': '-'
    },
    'SGD_Momentum': {
        'color': '#1F77B4',
        'linestyle': '--'
    },
    'RMSprop': {
        'color': '#2CA02C',
        'linestyle': '-.'
    },
    'Adam_Default': {
        'color': '#9467BD',
        'linestyle': ':'
    },
    'Adam_Explicit': {
        'color': '#E67E22',
        'linestyle': (0, (5, 2))
    }
}

plt.figure(figsize=(14, 8))

for name, history in optimizer_histories.items():

    val_accuracy = np.array(
        history.history['val_accuracy']
    )

    style = styles[name]

    plt.plot(
        epochs,
        val_accuracy,
        color=style['color'],
        linestyle=style['linestyle'],
        linewidth=2,
        label=name
    )

    crossed_epochs = np.where(val_accuracy >= 0.83)[0]

    if len(crossed_epochs) > 0:

        first_epoch_index = crossed_epochs[0]
        first_epoch = first_epoch_index + 1
        first_accuracy = val_accuracy[first_epoch_index]

        plt.annotate(
            f'{name}\nEpoch {first_epoch}',
            xy=(first_epoch, first_accuracy),
            xytext=(first_epoch, first_accuracy + 0.045),
            ha='center',
            fontsize=8,
            color=style['color'],
            arrowprops=dict(
                arrowstyle='->',
                color=style['color'],
                lw=1.5
            )
        )

    else:
        print(f"{name} did not reach 83% validation accuracy.")

plt.axhline(
    0.80,
    color='gray',
    linestyle='--',
    linewidth=1.5,
    label='80% Reference'
)

plt.axhline(
    0.83,
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='83% Threshold'
)

plt.axhline(
    0.86,
    color='gray',
    linestyle=':',
    linewidth=1.5,
    label='86% Reference'
)

plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.title(
    'Optimizer Comparison - Validation Accuracy over 50 Epochs (Adult Income ANN)'
)

plt.xticks(np.arange(0, 51, 5))
plt.ylim(0.70, 1.00)
plt.legend(loc='best')
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

### 7.4 - Learning rate sensitivity - SGD vs Adam

In [ ]:
lr_experiments = {
    'SGD_lr_0.0001': keras.optimizers.SGD(learning_rate=0.0001),
    'SGD_lr_0.001': keras.optimizers.SGD(learning_rate=0.001),
    'SGD_lr_0.01': keras.optimizers.SGD(learning_rate=0.01),
    'SGD_lr_0.1': keras.optimizers.SGD(learning_rate=0.1),

    'Adam_lr_0.0001': keras.optimizers.Adam(learning_rate=0.0001),
    'Adam_lr_0.001': keras.optimizers.Adam(learning_rate=0.001),
    'Adam_lr_0.01': keras.optimizers.Adam(learning_rate=0.01)
}

lr_models = {}
lr_histories = {}

for name, optimizer in lr_experiments.items():

    model = build_ann(
        input_dim=X_train.shape[1],
        activation='relu',
        initializer='he_normal',
        use_batch_norm=True,
        loss='binary_crossentropy',
        optimizer=optimizer
    )

    history = model.fit(
        X_train,
        y_train,
        epochs=30,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )

    lr_models[name] = model
    lr_histories[name] = history

In [ ]:
epochs = np.arange(1, 31)

sgd_lrs = ['0.0001', '0.001', '0.01', '0.1']
adam_lrs = ['0.0001', '0.001', '0.01']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for lr in sgd_lrs:
    name = f'SGD_lr_{lr}'

    axes[0].plot(
        epochs,
        lr_histories[name].history['val_loss'],
        linewidth=2,
        label=f'lr = {lr}'
    )

axes[0].set_title('Vanilla SGD — Learning Rate Sensitivity')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Validation Loss')
axes[0].set_xticks(np.arange(0, 31, 5))
axes[0].legend()
axes[0].grid(alpha=0.3)


for lr in adam_lrs:
    name = f'Adam_lr_{lr}'

    axes[1].plot(
        epochs,
        lr_histories[name].history['val_loss'],
        linewidth=2,
        label=f'lr = {lr}'
    )

axes[1].set_title('Adam — Learning Rate Sensitivity')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Validation Loss')
axes[1].set_xticks(np.arange(0, 31, 5))
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle(
    'Learning Rate Sensitivity - SGD vs Adam',
    fontsize=15,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
final_val_losses = {
    name: history.history['val_loss'][-1]
    for name, history in lr_histories.items()
}

for name, loss in final_val_losses.items():
    print(f'{name}: {loss:.4f}')

best_sgd = min(
    sgd_lrs,
    key=lambda lr: final_val_losses[f'SGD_lr_{lr}']
)

best_adam = min(
    adam_lrs,
    key=lambda lr: final_val_losses[f'Adam_lr_{lr}']
)

print(f'\nBest SGD learning rate: {best_sgd}')
print(f'Best Adam learning rate: {best_adam}')

### 7.5 - Final Combined ANN - best configuration

In [ ]:
use_weighted_bce = False

final_class_weight = class_weight_dict if use_weighted_bce else None

final_model = build_ann(
    input_dim=X_train.shape[1],
    hidden_units=[128, 64],
    activation='relu',
    initializer='he_normal',
    use_batch_norm=True,
    optimizer='adam',
    loss='binary_crossentropy'
)

history_final = final_model.fit(
    X_train,
    y_train,
    epochs=80,
    batch_size=64,
    validation_split=0.1,
    class_weight=final_class_weight,
    verbose=1
)

In [ ]:
y_test_proba = final_model.predict(X_test, verbose=0).ravel()

y_test_pred = (y_test_proba >= 0.5).astype(int)

final_accuracy = accuracy_score(y_test, y_test_pred)

final_precision = precision_score(
    y_test, y_test_pred, pos_label=1
)

final_recall = recall_score(
    y_test, y_test_pred, pos_label=1
)

final_f1 = f1_score(
    y_test, y_test_pred, pos_label=1
)

final_roc_auc = roc_auc_score(
    y_test, y_test_proba
)

final_results = pd.DataFrame({
    'Metric': [
        'Accuracy',
        'Precision (Class 1)',
        'Recall (Class 1)',
        'F1-Score (Class 1)',
        'ROC-AUC'
    ],
    'Final ANN': [
        final_accuracy,
        final_precision,
        final_recall,
        final_f1,
        final_roc_auc
    ]
})

print("Final ANN Test Performance")
display(final_results.round(4))

In [ ]:
final_model.summary()

### 7.6 - Full results comparison table

In [ ]:
activation_models = {
    'relu': model_relu,
    'tanh': model_tanh,
    'sigmoid': model_sigmoid,
    'elu': model_elu
}

activation_histories = {
    'relu': history_relu,
    'tanh': history_tanh,
    'sigmoid': history_sigmoid,
    'elu': history_elu
}

# Best activation
best_activation = max(
    activation_histories,
    key=lambda name: activation_histories[name].history['val_f1_score'][-1]
)
best_activation_model = activation_models[best_activation]

# Best initializer
best_initializer = max(
    initializer_histories,
    key=lambda name: initializer_histories[name].history['val_f1_score'][-1]
)
best_initializer_model = initializer_models[best_initializer]

print("Best Activation:", best_activation)
print("Best Initializer:", best_initializer)

In [ ]:
models_to_compare = {
    'Baseline (Task 2)': {
        'model': model_base,
        'activation': 'relu',
        'init': 'glorot_uniform',
        'loss': 'binary_crossentropy',
        'batch_norm': False,
        'optimizer': 'adam'
    },

    'Best Activation (Task 3)': {
        'model': best_activation_model,
        'activation': best_activation,
        'init': 'glorot_uniform',
        'loss': 'binary_crossentropy',
        'batch_norm': False,
        'optimizer': 'adam'
    },

    'Best Initialiser (Task 4)': {
        'model': best_initializer_model,
        'activation': 'relu',
        'init': best_initializer,
        'loss': 'binary_crossentropy',
        'batch_norm': False,
        'optimizer': 'adam'
    },

    'Weighted BCE (Task 5)': {
        'model': model_wbce,
        'activation': 'relu',
        'init': 'he_normal',
        'loss': 'weighted_binary_crossentropy',
        'batch_norm': False,
        'optimizer': 'adam'
    },

    'Focal Loss (Task 5)': {
        'model': model_focal,
        'activation': 'relu',
        'init': 'he_normal',
        'loss': 'focal_loss',
        'batch_norm': False,
        'optimizer': 'adam'
    },

    'BatchNorm (Task 6)': {
        'model': model_bn,
        'activation': 'relu',
        'init': 'he_normal',
        'loss': 'binary_crossentropy',
        'batch_norm': True,
        'optimizer': 'adam'
    },

    'Final Combined (Task 7)': {
        'model': final_model,
        'activation': 'relu',
        'init': 'he_normal',
        'loss': 'weighted_binary_crossentropy' if use_weighted_bce
                else 'binary_crossentropy',
        'batch_norm': True,
        'optimizer': 'adam'
    }
}

# Calculate test metrics for every model
comparison_rows = []
test_probabilities = {}

for model_name, config in models_to_compare.items():

    model = config['model']

    # Predicted probabilities and class labels
    y_proba = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_proba >= 0.5).astype(int)

    test_probabilities[model_name] = y_proba

    comparison_rows.append({
        'Model': model_name,
        'Activation': config['activation'],
        'Init': config['init'],
        'Loss': config['loss'],
        'BatchNorm': config['batch_norm'],
        'Optimizer': config['optimizer'],
        'Test Acc': accuracy_score(y_test, y_pred),
        'Prec(1)': precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        'Recall(1)': recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        'F1(1)': f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

# Create the comparison DataFrame
comparison_df = pd.DataFrame(comparison_rows)

# Display highlighted results
display(
    comparison_df.style
    .format({
        'Test Acc': '{:.4f}',
        'Prec(1)': '{:.4f}',
        'Recall(1)': '{:.4f}',
        'F1(1)': '{:.4f}',
        'ROC-AUC': '{:.4f}'
    })
    .highlight_max(
        subset=['F1(1)', 'ROC-AUC'],
        color='#ffcccc'
    )
)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

roc_models = [
    'Baseline (Task 2)',
    'Weighted BCE (Task 5)',
    'Final Combined (Task 7)'
]

plt.figure(figsize=(10, 7))

for model_name in roc_models:

    y_proba = test_probabilities[model_name]

    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f'{model_name} (AUC = {auc_score:.4f})'
    )

# Random classifier reference
plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    color='gray',
    label='Random Classifier'
)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Baseline vs Weighted BCE vs Final Combined')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()